In [0]:
from pyspark.sql import functions as F

# Read Silver tables
train = spark.table(
    "imdb_sentiment.silver.train_reviews"
)

test = spark.table(
    "imdb_sentiment.silver.test_reviews"
)

# Identify overlapping review texts
overlap_texts = (
    train.select("clean_text").distinct()
    .join(
        test.select("clean_text").distinct(),
        on="clean_text",
        how="inner"
    )
)

# Remove overlapping reviews from training
train_deduplicated = train.join(
    overlap_texts,
    on="clean_text",
    how="left_anti"
)

print("Training rows:", train_deduplicated.count())
print("Test rows:", test.count())
print("Overlapping texts:", overlap_texts.count())

Training rows: 24877
Test rows: 25000
Overlapping texts: 123


In [0]:
# Prepare model-ready training data
gold_train = train_deduplicated.select(
    "clean_text",
    "label",
    "text_length"
)

# Prepare model-ready test data
gold_test = test.select(
    "clean_text",
    "label",
    "text_length"
)

# Save as Delta tables
gold_train.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("imdb_sentiment.gold.train_reviews")

gold_test.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("imdb_sentiment.gold.test_reviews")

print("Gold tables created successfully!")
print("Training rows:", gold_train.count())
print("Test rows:", gold_test.count())

Gold tables created successfully!
Training rows: 24877
Test rows: 25000


In [0]:
from pyspark.sql import functions as F

gold_train = spark.table(
    "imdb_sentiment.gold.train_reviews"
)

gold_test = spark.table(
    "imdb_sentiment.gold.test_reviews"
)

print("TRAINING DISTRIBUTION")
display(
    gold_train.groupBy("label")
    .count()
    .orderBy("label")
)

print("TEST DISTRIBUTION")
display(
    gold_test.groupBy("label")
    .count()
    .orderBy("label")
)

print("TRAIN/TEST OVERLAP")

overlap_count = (
    gold_train.select("clean_text").distinct()
    .join(
        gold_test.select("clean_text").distinct(),
        on="clean_text",
        how="inner"
    )
    .count()
)

print("Shared unique review texts:", overlap_count)

print("GOLD TRAIN SCHEMA")
gold_train.printSchema()

TRAINING DISTRIBUTION


label,count
0,12405
1,12472


TEST DISTRIBUTION


label,count
0,12500
1,12500


TRAIN/TEST OVERLAP
Shared unique review texts: 0
GOLD TRAIN SCHEMA
root
 |-- clean_text: string (nullable = true)
 |-- label: long (nullable = true)
 |-- text_length: integer (nullable = true)



In [0]:
from pyspark.sql import functions as F

gold_train = spark.table("imdb_sentiment.gold.train_reviews")
gold_test = spark.table("imdb_sentiment.gold.test_reviews")

print("TEST DISTRIBUTION")

display(
    gold_test.groupBy("label")
    .count()
    .orderBy("label")
)

print("TRAIN / TEST OVERLAP")

overlap_count = (
    gold_train.select("clean_text").distinct()
    .join(
        gold_test.select("clean_text").distinct(),
        on="clean_text",
        how="inner"
    )
    .count()
)

print("Shared unique review texts:", overlap_count)

TEST DISTRIBUTION


label,count
0,12500
1,12500


TRAIN / TEST OVERLAP
Shared unique review texts: 0


In [0]:
display(
    spark.sql("SHOW VOLUMES IN imdb_sentiment.gold")
)

database,volume_name
gold,ml_exports


In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS
imdb_sentiment.gold.ml_exports
""")

print("Export Volume created successfully!")

display(
    spark.sql("SHOW VOLUMES IN imdb_sentiment.gold")
)

Export Volume created successfully!


database,volume_name
gold,ml_exports


In [0]:
# Read Gold tables
gold_train = spark.table(
    "imdb_sentiment.gold.train_reviews"
)

gold_test = spark.table(
    "imdb_sentiment.gold.test_reviews"
)

# Export training dataset
gold_train.coalesce(1).write \
    .mode("overwrite") \
    .parquet(
        "/Volumes/imdb_sentiment/gold/ml_exports/train"
    )

# Export test dataset
gold_test.coalesce(1).write \
    .mode("overwrite") \
    .parquet(
        "/Volumes/imdb_sentiment/gold/ml_exports/test"
    )

print("Parquet exports completed successfully!")

Parquet exports completed successfully!


In [0]:
export_path = "/Volumes/imdb_sentiment/gold/ml_exports"

print("Exported files:")

display(
    dbutils.fs.ls(export_path)
)

print("Training folder:")
display(
    dbutils.fs.ls(f"{export_path}/train")
)

print("Test folder:")
display(
    dbutils.fs.ls(f"{export_path}/test")
)

Exported files:


path,name,size,modificationTime
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/,test/,0,1793362425156
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test_csv/,test_csv/,0,1793362425156
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/,train/,0,1793362425156
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train_csv/,train_csv/,0,1793362425156


Training folder:


path,name,size,modificationTime
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/_SUCCESS,_SUCCESS,0,1793362423000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/_committed_1864097744683179008,_committed_1864097744683179008,124,1793360383000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/_committed_5583464658689526033,_committed_5583464658689526033,223,1793362423000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/_committed_7278075779928071603,_committed_7278075779928071603,234,1793361319000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/_committed_vacuum8140509198444979310,_committed_vacuum8140509198444979310,96,1793362423000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/_started_5583464658689526033,_started_5583464658689526033,0,1793362422000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/_started_7278075779928071603,_started_7278075779928071603,0,1793361318000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/part-00000-tid-5583464658689526033-35546156-4591-4fc4-9fc5-6512fe65c46b-514-1.c000.snappy.parquet,part-00000-tid-5583464658689526033-35546156-4591-4fc4-9fc5-6512fe65c46b-514-1.c000.snappy.parquet,21053582,1793362422000


Test folder:


path,name,size,modificationTime
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/_SUCCESS,_SUCCESS,0,1793362425000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/_committed_1811623700416949565,_committed_1811623700416949565,234,1793361321000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/_committed_7347668655326685243,_committed_7347668655326685243,124,1793360385000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/_committed_8787485059019987809,_committed_8787485059019987809,223,1793362424000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/_committed_vacuum5847163503883291573,_committed_vacuum5847163503883291573,96,1793362425000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/_started_1811623700416949565,_started_1811623700416949565,0,1793361320000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/_started_8787485059019987809,_started_8787485059019987809,0,1793362424000
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/part-00000-tid-8787485059019987809-1789a2d0-92b4-4ffe-9f0e-f64bce377086-515-1.c000.snappy.parquet,part-00000-tid-8787485059019987809-1789a2d0-92b4-4ffe-9f0e-f64bce377086-515-1.c000.snappy.parquet,20010719,1793362424000


In [0]:
export_path = "/Volumes/imdb_sentiment/gold/ml_exports"

train_file = [
    f.path for f in dbutils.fs.ls(f"{export_path}/train")
    if f.name.endswith(".parquet")
][0]

test_file = [
    f.path for f in dbutils.fs.ls(f"{export_path}/test")
    if f.name.endswith(".parquet")
][0]

print("Training file:", train_file)
print("Test file:", test_file)

Training file: dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train/part-00000-tid-5583464658689526033-35546156-4591-4fc4-9fc5-6512fe65c46b-514-1.c000.snappy.parquet
Test file: dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test/part-00000-tid-8787485059019987809-1789a2d0-92b4-4ffe-9f0e-f64bce377086-515-1.c000.snappy.parquet


In [0]:
export_path = "/Volumes/imdb_sentiment/gold/ml_exports"

gold_train = spark.table("imdb_sentiment.gold.train_reviews")

gold_train.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .csv(f"{export_path}/train_csv")

print("Training dataset exported as CSV!")

Training dataset exported as CSV!


In [0]:
gold_test = spark.table("imdb_sentiment.gold.test_reviews")

gold_test.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .csv(f"{export_path}/test_csv")

print("Test dataset exported as CSV!")

Test dataset exported as CSV!


In [0]:
train_csv_path = f"{export_path}/train_csv"
test_csv_path = f"{export_path}/test_csv"

print("TRAIN CSV FILES:")
for file in dbutils.fs.ls(train_csv_path):
    print(file.path, "|", file.size, "bytes")

print("\nTEST CSV FILES:")
for file in dbutils.fs.ls(test_csv_path):
    print(file.path, "|", file.size, "bytes")

TRAIN CSV FILES:
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train_csv/_SUCCESS | 0 bytes
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train_csv/_committed_3305660696203847776 | 113 bytes
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train_csv/_committed_5808426902174289827 | 212 bytes
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train_csv/_started_3305660696203847776 | 0 bytes
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train_csv/_started_5808426902174289827 | 0 bytes
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/train_csv/part-00000-tid-5808426902174289827-9a969ed1-5b83-49b0-a6a5-9aca2825f05b-552-1-c000.csv | 32758330 bytes

TEST CSV FILES:
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test_csv/_SUCCESS | 0 bytes
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test_csv/_committed_2984885562546506418 | 212 bytes
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test_csv/_committed_3209757009074957668 | 113 bytes
dbfs:/Volumes/imdb_sentiment/gold/ml_exports/test_csv/_started_2984885562546506418

In [0]:
gold_train = spark.table("imdb_sentiment.gold.train_reviews")

gold_train.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(f"{export_path}/train_csv_v2")

print("Training CSV v2 exported!")

Training CSV v2 exported!


In [0]:
gold_test = spark.table("imdb_sentiment.gold.test_reviews")

gold_test.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("encoding", "UTF-8") \
    .option("quote", '"') \
    .option("escape", '"') \
    .csv(f"{export_path}/test_csv_v2")

print("Test CSV v2 exported!")

Test CSV v2 exported!
